# NLLB-200 3.3B with CTranslate2 (Low VRAM Inference)

Fast and memory-efficient French ↔ Kabyle translation using quantized NLLB-200 with CTranslate2.

In [ ]:
# 1. Install Dependencies
!pip install ctranslate2 transformers sentencepiece huggingface_hub safetensors

# Verify versions
import torch
import transformers
import ctranslate2

print(f"\n{'='*50}")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CTranslate2 version: {ctranslate2.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"{'='*50}")

In [ ]:
# 2. Check CUDA availability
import ctranslate2

print(f"CTranslate2 version: {ctranslate2.__version__}")
print(f"CUDA available: {ctranslate2.get_cuda_device_count() > 0}")
print(f"Number of CUDA devices: {ctranslate2.get_cuda_device_count()}")

# Check supported compute types
print(
    f"\nSupported compute types on CUDA: {ctranslate2.get_supported_compute_types('cuda')}")

## Model Conversion (One-Time Setup)

You need to convert the NLLB model to CTranslate2 format. This is done once and saved to disk.

Choose your quantization based on available VRAM:
- `float16` - Best quality, ~6.5 GB VRAM
- `int8_float16` - Good quality, ~3.5 GB VRAM (recommended)
- `int8` - Fastest, lowest VRAM, slight quality reduction

In [ ]:
# 3. Download pre-converted CTranslate2 model (recommended)
# Instead of converting ourselves, use a pre-converted model from HuggingFace
import os
from huggingface_hub import snapshot_download

CT2_MODEL_PATH = "./nllb-200-3.3B-ct2"

# Pre-converted NLLB models on HuggingFace (CTranslate2 format)
# Using JustFrederik's conversion - well-maintained and popular
CT2_MODEL_REPO = "JustFrederik/nllb-200-3.3B-ct2-float16"  # float16 version

if not os.path.exists(CT2_MODEL_PATH):
    print(f"Downloading pre-converted CTranslate2 model...")
    print(f"Repository: {CT2_MODEL_REPO}")
    print("This will download ~6.5 GB.\n")

    local_dir = snapshot_download(
        repo_id=CT2_MODEL_REPO,
        local_dir=CT2_MODEL_PATH,
    )
    print(f"\n✅ Model downloaded to: {local_dir}")
else:
    print(f"✅ Model already exists at: {CT2_MODEL_PATH}")

# List contents to verify
print("\nModel files:")
for f in os.listdir(CT2_MODEL_PATH):
    size = os.path.getsize(os.path.join(CT2_MODEL_PATH, f)) / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

In [ ]:
# 4. Load the CTranslate2 model
import torch
import ctranslate2
from transformers import AutoTokenizer
import subprocess

CT2_MODEL_PATH = "./nllb-200-3.3B-ct2"
MODEL_NAME = "facebook/nllb-200-3.3B"

# Load tokenizer from original model
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load CTranslate2 model
print("Loading CTranslate2 model on CUDA...")
translator = ctranslate2.Translator(
    CT2_MODEL_PATH,
    device="cuda",
    device_index=0,  # GPU index
    compute_type="float16",  # Use float16 (matches the downloaded model)
    inter_threads=1,
    intra_threads=4,
)

print("✅ Model loaded successfully!")

# Check actual GPU memory using nvidia-smi (CTranslate2 doesn't use PyTorch's memory pool)
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total",
            "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        used, total = result.stdout.strip().split(", ")
        print(f"\nGPU Memory (nvidia-smi): {used} MB / {total} MB")
except:
    pass

In [ ]:
# 5. Define translation function

# NLLB language codes
LANG_CODES = {
    "french": "fra_Latn",
    "kabyle": "kab_Latn",
    "english": "eng_Latn",
    "arabic": "arb_Arab",
    "spanish": "spa_Latn",
    "german": "deu_Latn",
}


def translate(
    text: str | list[str],
    source_lang: str = "french",
    target_lang: str = "kabyle",
    beam_size: int = 5,
    max_length: int = 256,
) -> list[str]:
    """
    Translate text using CTranslate2 NLLB model.

    Args:
        text: Single string or list of strings to translate
        source_lang: Source language name (e.g., "french", "english")
        target_lang: Target language name (e.g., "kabyle", "english")
        beam_size: Beam search size (higher = better quality, slower)
        max_length: Maximum output length

    Returns:
        List of translated strings
    """
    # Handle single string input
    if isinstance(text, str):
        text = [text]

    # Get language codes
    src_lang = LANG_CODES.get(source_lang.lower(), source_lang)
    tgt_lang = LANG_CODES.get(target_lang.lower(), target_lang)

    # Tokenize with source language prefix
    tokenizer.src_lang = src_lang

    # Tokenize all inputs
    source_tokens = []
    for t in text:
        tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(t))
        source_tokens.append(tokens)

    # Translate
    target_prefix = [[tgt_lang]] * len(text)

    results = translator.translate_batch(
        source_tokens,
        target_prefix=target_prefix,
        beam_size=beam_size,
        max_decoding_length=max_length,
    )

    # Decode results
    translations = []
    for result in results:
        # Get tokens and skip the language code prefix
        tokens = result.hypotheses[0]
        if tokens and tokens[0] == tgt_lang:
            tokens = tokens[1:]

        # Decode tokens to text
        translation = tokenizer.decode(
            tokenizer.convert_tokens_to_ids(tokens),
            skip_special_tokens=True
        )
        translations.append(translation)

    return translations


print("✅ Translation function defined!")

In [ ]:
# 6. Test translations - French to Kabyle

test_sentences = [
    "Bonjour, comment vas-tu ?",
    "La langue kabyle est très belle.",
    "Où est ta maison ?",
    "Je t'aime beaucoup.",
    "Le soleil brille aujourd'hui.",
]

print("French → Kabyle Translation")
print("=" * 60)

# Translate batch (more efficient)
translations = translate(
    test_sentences, source_lang="french", target_lang="kabyle")

for fr, kab in zip(test_sentences, translations):
    print(f"FR:  {fr}")
    print(f"KAB: {kab}")
    print("-" * 40)

In [ ]:
# 7. Benchmark: Compare speed
import time

# Warmup
_ = translate("Test warmup sentence.")

# Benchmark single sentence
test_text = "La langue kabyle est une langue berbère parlée en Kabylie, une région montagneuse d'Algérie."

start = time.perf_counter()
for _ in range(10):
    result = translate(test_text)
elapsed = time.perf_counter() - start

print(f"Single sentence (avg of 10): {elapsed/10*1000:.1f} ms")
print(f"Translation: {result[0]}")

# Benchmark batch
batch = [test_text] * 8
start = time.perf_counter()
results = translate(batch)
elapsed = time.perf_counter() - start

print(f"\nBatch of 8 sentences: {elapsed*1000:.1f} ms total")
print(f"Per sentence in batch: {elapsed/8*1000:.1f} ms")

In [ ]:
# 8. Interactive translation
def interactive_translate():
    """Interactive translation loop."""
    print("=" * 60)
    print("CTranslate2 NLLB Translator")
    print("Default: French → Kabyle")
    print("Commands: 'quit' to exit, 'lang' to change languages")
    print("=" * 60)

    src, tgt = "french", "kabyle"

    while True:
        user_input = input(f"\n[{src} → {tgt}] Enter text: ").strip()

        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        elif user_input.lower() == 'lang':
            print(f"Available: {list(LANG_CODES.keys())}")
            src = input("Source language: ").strip().lower()
            tgt = input("Target language: ").strip().lower()
            continue
        elif user_input:
            start = time.perf_counter()
            result = translate(user_input, source_lang=src, target_lang=tgt)
            elapsed = (time.perf_counter() - start) * 1000
            print(f"→ {result[0]}  ({elapsed:.0f} ms)")

# Uncomment to run:
# interactive_translate()

## Tips for Even Lower VRAM

### 1. Use a Smaller Model
If 3.3B is still too large, try:
- `facebook/nllb-200-1.3B` (~1.5 GB with int8)
- `facebook/nllb-200-distilled-600M` (~700 MB with int8)

### 2. CPU Offloading
```python
translator = ctranslate2.Translator(
    CT2_MODEL_PATH,
    device="cuda",
    device_index=0,
    compute_type="int8",
    # Limit GPU memory usage
)
```

### 3. Use CPU with Multiple Threads
```python
translator = ctranslate2.Translator(
    CT2_MODEL_PATH,
    device="cpu",
    compute_type="int8",
    inter_threads=2,  # Parallel requests
    intra_threads=8,  # Threads per request
)
```

### 4. Batch Processing
Always batch your translations when possible - it's much more efficient!

In [ ]:
# 9. Memory cleanup (run when done)
import gc
import torch


def cleanup():
    """Free GPU memory."""
    global translator
    del translator
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("✅ Memory cleaned up!")

# Uncomment to free memory:
# cleanup()